# 01 Best PseudoGT Learnability

This notebook separates pseudo-GT learning from DQA aggregation.
Every client starts from the copied 08_4 warmup checkpoint, then
candidate pseudo-GT training profiles are compared on the same
scene protocol used by the DQA experiments.


## Fixed condition

- Start checkpoint: `checkpoints/round000_warmup.pt`
- Output root: `output/`
- Clients: `highway`, `citystreet`, `residential`
- Evaluation: `highway`, `citystreet`, `residential`, `total`
- Main question: can pseudo-GT training itself produce useful
  client updates before DQA-style class-wise aggregation is added?


In [2]:
from pathlib import Path
import os
import subprocess
import sys

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "pseudogt_learnability").exists():
    PROJECT_ROOT = cwd / "pseudogt_learnability"
else:
    PROJECT_ROOT = cwd
os.chdir(PROJECT_ROOT)
print(PROJECT_ROOT)
print((PROJECT_ROOT / "checkpoints" / "round000_warmup.pt").exists())


/app/Object_Detection/pseudogt_learnability
True


## Setup check

This creates/reuses scene data lists in `output/` and verifies that
the copied warmup checkpoint is visible from the new project.


In [3]:
subprocess.run([
    sys.executable,
    "scripts/run_pseudogt_learnability_01.py",
    "--setup-only",
], check=True)


{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "6f985a45c8432cc40a8362ed44d1f7a8498bd5d6",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "6f985a45c8432cc40a8362ed44d1f7a8498bd5d6",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/pseudogt_learnability/output/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/pseudogt_learnability/output/data_lists/paper_eval_scene_total_val.txt",
      "source_val_list": "/app/Object_Detection/pseudogt_learnability/output/data_lists/server_cloudy_val.txt",
      "train_images": 4881,
      "val_images": 9864,
      "source_val_images":

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', 'scripts/run_pseudogt_learnability_01.py', '--setup-only'], returncode=0)

## Run 01

The default run tests three profiles:

- `backbone_obj_safe`
- `neck_head_high_precision`
- `all_consistency_lowlr`

It saves client checkpoints, aggregate checkpoints, one-epoch
server-repair checkpoints, and then evaluates them scene-wise.


In [4]:
cmd = [
    sys.executable,
    "scripts/run_pseudogt_learnability_01.py",
    "--variants", "backbone_obj_safe,neck_head_high_precision,all_consistency_lowlr",
    "--batch-size", "160",
    "--workers", "8",
    "--gpus", "2",
    "--master-port", "30341",
    "--server-repair-epochs", "1",
    "--evaluate",
    "--classwise",
    "--no-eval-plots",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


/root/micromamba/envs/al_yolov8/bin/python scripts/run_pseudogt_learnability_01.py --variants backbone_obj_safe,neck_head_high_precision,all_consistency_lowlr --batch-size 160 --workers 8 --gpus 2 --master-port 30341 --server-repair-epochs 1 --evaluate --classwise --no-eval-plots
{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "6f985a45c8432cc40a8362ed44d1f7a8498bd5d6",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "6f985a45c8432cc40a8362ed44d1f7a8498bd5d6",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/pseudogt_learnability/output/data_lists/server_cloudy_train.txt",
      "val_list": "/app/O


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


train: __immutable__=False, __deprecated_keys__=set(), __renamed_keys__={}, __new_allowed__=False
Weights & Biases: run 'pip install wandb' to automatically track and visualize EfficientTeacher runs (RECOMMENDED)


EfficientTeacher  6f985a4 torch 2.8.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

TensorBoard: Start with 'tensorboard --logdir /app/Object_Detection/pseudogt_learnability/output/runs', view at http://localhost:6006/
Model summary: 478 layers, 47566599 parameters, 47566599 gradients

Transferred 612/619 items from /app/Object_Detection/pseudogt_learnability/output/client_states/pl01_backbone_obj_safe_client0_highway_start.pt
Scaled weight_decay = 0.00125
optimizer: SGD with parameter groups 110 weight, 101 weight (no decay), 104 bias
/app/Object_Detection/navigating_data_heterogeneity/vendor/efficientteacher/trainer/trainer.py:251: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = amp.GradScaler(enabled=self.cuda)
/app/Object_Detection/navigating_data_heterogeneity/vendor/efficientteacher/trainer/trainer.py:251: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated.

world_size: 2
rank: 0
world_size: 2
rank: 1


...


Validating /app/Object_Detection/pseudogt_learnability/output/runs/pl01_all_consistency_lowlr_server_repair/weights/best.pt...
Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 5/5 [00:04<00:00,  1.09it/s]


                 all        738      14937      0.475      0.431        0.4      0.224
                   0        738       1052      0.481      0.559      0.536      0.255
                   1        738         44      0.337      0.136      0.142     0.0561
                   2        738       8622      0.588      0.756      0.751      0.483
                   3        738        151      0.351      0.411      0.384      0.294
                   4        738        467      0.465      0.593      0.547      0.394
                   5        738         70      0.268      0.457      0.277      0.128
                   6        738         65       0.21      0.123      0.171     0.0657
                   7        738       1619      0.514      0.631      0.584      0.242
                   8        738       2845      0.535      0.643      0.611       0.32
                   9        738          2          1          0   0.000307   0.000276


Results saved to /app/Object_Detection/pseudogt_learnability/output/runs/pl01_all_consistency_lowlr_server_repair

1 epochs completed in 0.028 hours.
Destroying process group... 
[rank0]:[W505 02:37:27.495293841 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Saved: /app/Object_Detection/pseudogt_learnability/output/stats/01_checkpoints.csv
Saved: /app/Object_Detection/pseudogt_learnability/output/stats/01_manifest.json
/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/pseudogt_learnability/scripts/evaluate_scene_protocol.py --workspace /app/Object_Detection/pseudogt_learnability/output --splits highway,citystreet,residential,total --batch-size 16 --no-plots --verbose --checkpoint warmup_global=/app/Object_Detection/pseudogt_learnability/output/global_checkpoints/round000_warmup.pt --checkpoint backbone_obj_safe_client0_highway=/app/Object_Detection/pseudogt_learnability/output/checkpoints/backbone_obj_safe_client0_highway.pt --checkpoint backbone_obj_safe_client1_citystreet=/app/Object_Detection/pseudogt_learnability/output/checkpoints/backbone_obj_safe_client1_citystreet.pt --checkpoint backbone_obj_safe_client2_residential=/app/Object_Detection/pseudogt_learnability/output/checkpoints/backbone_obj_safe_client2_residential.

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', 'scripts/run_pseudogt_learnability_01.py', '--variants', 'backbone_obj_safe,neck_head_high_precision,all_consistency_lowlr', '--batch-size', '160', '--workers', '8', '--gpus', '2', '--master-port', '30341', '--server-repair-epochs', '1', '--evaluate', '--classwise', '--no-eval-plots'], returncode=0)

## Check generated checkpoints


In [5]:
import pandas as pd

checkpoint_table = PROJECT_ROOT / "output" / "stats" / "01_checkpoints.csv"
if checkpoint_table.exists():
    ckpts = pd.read_csv(checkpoint_table)
    display(ckpts)
else:
    print("No checkpoint table yet:", checkpoint_table)


,label,kind,variant,client,path
0,warmup_global,warmup,NaN,NaN,/app/Object_Detection/pseudogt_learnability/ou...
1,backbone_obj_safe_client0_highway,client,backbone_obj_safe,client0_highway,/app/Object_Detection/pseudogt_learnability/ou...
2,backbone_obj_safe_client1_citystreet,client,backbone_obj_safe,client1_citystreet,/app/Object_Detection/pseudogt_learnability/ou...
3,backbone_obj_safe_client2_residential,client,backbone_obj_safe,client2_residential,/app/Object_Detection/pseudogt_learnability/ou...
4,backbone_obj_safe_aggregate_backbone,aggregate,backbone_obj_safe,NaN,/app/Object_Detection/pseudogt_learnability/ou...
5,backbone_obj_safe_server_repair,server_repair,backbone_obj_safe,NaN,/app/Object_Detection/pseudogt_learnability/ou...
6,neck_head_high_precision_client0_highway,client,neck_head_high_precision,client0_highway,/app/Object_Detection/pseudogt_learnability/ou...
7,neck_head_high_precision_client1_citystreet,client,neck_head_high_precision,client1_citystreet,/app/Object_Detection/pseudogt_learnability/ou...
8,neck_head_high_precision_client2_residential,client,neck_head_high_precision,client2_residential,/app/Object_Detection/pseudogt_learnability/ou...
9,neck_head_high_precision_aggregate_all,aggregate,neck_head_high_precision,NaN,/app/Object_Detection/pseudogt_learnability/ou...


## Evaluation summary

The key metric is whether any pseudo-GT profile beats
`warmup_global` on `total` mAP@0.5:0.95, or at least improves the
matching client scene without damaging the total split.


In [6]:
import pandas as pd

summary_csv = PROJECT_ROOT / "output" / "validation_reports" / "paper_protocol_eval_summary.csv"
if not summary_csv.exists():
    print("Evaluation summary is not available yet:", summary_csv)
else:
    df = pd.read_csv(summary_csv)
    ok = df[df["status"].eq("ok")].copy()
    total = ok[ok["split"].isin(["total", "scene_total"])].copy()
    total = total.sort_values("map50_95", ascending=False)
    display(total[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])

    warm = total[total["checkpoint_label"].eq("warmup_global")]
    if not warm.empty:
        base = float(warm.iloc[0]["map50_95"])
        total["delta_map50_95_vs_warmup"] = total["map50_95"].astype(float) - base
        display(total[["checkpoint_label", "map50", "map50_95", "delta_map50_95_vs_warmup"]])


,checkpoint_label,split,precision,recall,map50,map50_95
23,backbone_obj_safe_server_repair,scene_total,0.491,0.393,0.376,0.208
63,all_consistency_lowlr_server_repair,scene_total,0.493,0.393,0.376,0.208
43,neck_head_high_precision_server_repair,scene_total,0.492,0.393,0.376,0.208
19,backbone_obj_safe_aggregate_backbone,scene_total,0.461,0.381,0.351,0.189
3,warmup_global,scene_total,0.532,0.337,0.352,0.189
39,neck_head_high_precision_aggregate_all,scene_total,0.534,0.337,0.352,0.189
59,all_consistency_lowlr_aggregate_all,scene_total,0.534,0.338,0.352,0.189


,checkpoint_label,map50,map50_95,delta_map50_95_vs_warmup
23,backbone_obj_safe_server_repair,0.376,0.208,0.019
63,all_consistency_lowlr_server_repair,0.376,0.208,0.019
43,neck_head_high_precision_server_repair,0.376,0.208,0.019
19,backbone_obj_safe_aggregate_backbone,0.351,0.189,0.000
3,warmup_global,0.352,0.189,0.000
39,neck_head_high_precision_aggregate_all,0.352,0.189,0.000
59,all_consistency_lowlr_aggregate_all,0.352,0.189,0.000


## Scene-specific client learnability

This view is useful even if the aggregate is weak.  If a client
improves its own scene split, pseudo-GT may still be learnable, and
the aggregation rule is the next target.  If no client improves its
own split, the pseudo-GT training recipe itself is the bottleneck.


In [7]:
if summary_csv.exists():
    ok = pd.read_csv(summary_csv)
    ok = ok[ok["status"].eq("ok")].copy()
    scene_rows = ok[ok["split"].isin(["highway", "citystreet", "residential"])].copy()
    display(scene_rows.sort_values(["split", "map50_95"], ascending=[True, False])[
        ["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]
    ])


,checkpoint_label,split,precision,recall,map50,map50_95
21,backbone_obj_safe_server_repair,citystreet,0.490,0.398,0.381,0.210
41,neck_head_high_precision_server_repair,citystreet,0.491,0.399,0.381,0.210
61,all_consistency_lowlr_server_repair,citystreet,0.490,0.400,0.381,0.210
1,warmup_global,citystreet,0.528,0.342,0.357,0.191
17,backbone_obj_safe_aggregate_backbone,citystreet,0.461,0.387,0.357,0.191
37,neck_head_high_precision_aggregate_all,citystreet,0.533,0.342,0.357,0.191
57,all_consistency_lowlr_aggregate_all,citystreet,0.533,0.342,0.357,0.191
20,backbone_obj_safe_server_repair,highway,0.514,0.341,0.336,0.186
40,neck_head_high_precision_server_repair,highway,0.519,0.340,0.336,0.186
60,all_consistency_lowlr_server_repair,highway,0.519,0.340,0.336,0.186


## Discord notification


In [8]:
try:
    sys.path.insert(0, str(PROJECT_ROOT.parent))
    from notebook_notify import notify_discord

    msg = "PseudoGT Learnability 01 finished. Check output/validation_reports/paper_protocol_eval_summary.md"
    result = notify_discord(
        msg,
        title="PseudoGT Learnability 01",
        context={"project": str(PROJECT_ROOT)},
        fail_silently=True,
    )
    print(result)
except Exception as exc:
    print("Discord notification skipped:", exc)


DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)
